# HealthBot: AI-Powered Patient Education (LangGraph Prototype)

This notebook runs an interactive HealthBot workflow:
- asks for a health topic
- searches reputable sources via Tavily
- summarizes results in patient-friendly language with citations
- delivers a single-question quiz and feedback with citations


## Imports
Minimal imports for the workflow. Install dependencies with `pip install -r requirements.txt`.

In [ ]:
import json
import os
import re
import textwrap
from typing import Any, Dict, List, Optional, TypedDict

from dotenv import load_dotenv
from langchain_cohere import ChatCohere
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph
from rich.console import Console
from rich.panel import Panel


## Environment and tools
Keys, model, and search setup.

In [ ]:
load_dotenv(".env")

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not COHERE_API_KEY:
    raise RuntimeError("Missing COHERE_API_KEY")
if not TAVILY_API_KEY:
    raise RuntimeError("Missing TAVILY_API_KEY")

print("Environment loaded.")
console = Console()

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0.2,
    cohere_api_key=COHERE_API_KEY,
)

search_tool = TavilySearchResults(max_results=5, search_depth="advanced")


## State schema and helpers
State shape and small utility helpers.

In [ ]:
def print_panel(text: str, title: str, style: str = "cyan", width: int = 90) -> None:
    wrapped = "\n".join(textwrap.wrap(text, width=width))
    console.print(Panel.fit(wrapped, title=title, border_style=style))


def get_user_input(prompt: str, style: str = "bold yellow") -> str:
    answer = input(prompt + " ").strip()
    console.print(f"[{style}]You: {answer}[/{style}]")
    return answer


class HealthBotState(TypedDict, total=False):
    topic: Optional[str]
    search_results: Optional[List[Dict[str, Any]]]
    summary: Optional[str]
    quiz_question: Optional[str]
    quiz_answer: Optional[str]
    grade: Optional[str]
    grade_explanation: Optional[str]
    messages: List[Dict[str, str]]
    restart: Optional[bool]


def append_message(state: HealthBotState, role: str, content: str) -> List[Dict[str, str]]:
    messages = list(state.get("messages", []))
    messages.append({"role": role, "content": content})
    return messages


## Topic intake and search
Prompt the patient and call Tavily.

In [ ]:
def ask_topic(state: Dict[str, Any]) -> Dict[str, Any]:
    topic = get_user_input("What health topic or medical condition would you like to learn about? ").strip()
    return {
        "topic": topic,
        "messages": append_message(state, "user", f"Topic: {topic}"),
    }


def search_topic(state: Dict[str, Any]) -> Dict[str, Any]:
    topic = state.get("topic") or ""
    query = (
        f"{topic} (site:nih.gov OR site:cdc.gov OR site:mayoclinic.org "
        f"OR site:who.int OR site:medlineplus.gov)"
    )
    results = search_tool.invoke({"query": query})
    return {
        "search_results": results,
        "messages": append_message(state, "tool", f"Tavily search results for: {topic}"),
    }


## Summarization and readiness gate
Summarize sources and wait for readiness.

In [ ]:
def summarize_results(state: Dict[str, Any]) -> Dict[str, Any]:
    results = state.get("search_results") or []
    if not results:
        summary = "I couldn't find reliable results for that topic."
        return {
            "summary": summary,
            "messages": append_message(state, "assistant", summary),
        }

    sources_block = "\n\n".join(
        f"[{idx + 1}] {item.get('url', '')}\n{item.get('content', '')}"
        for idx, item in enumerate(results)
    )

    prompt = (
        "You are a patient education assistant. Using ONLY the sources below, "
        "write a clear, patient-friendly summary in 3-4 short paragraphs. "
        "Do not add outside knowledge. Include citations like [1], [2] tied to "
        "the sources provided.\n\n"
        f"SOURCES:\n{sources_block}"
    )

    response = llm.invoke(prompt)
    summary = (getattr(response, "content", None) or "").strip()

    return {
        "summary": summary,
        "messages": append_message(state, "assistant", summary),
    }


def present_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    print("\nSummary:\n")
    print(state.get("summary") or "")
    return {}


def wait_for_ready(state: Dict[str, Any]) -> Dict[str, Any]:
    get_user_input("\nWhen you're ready for a quick comprehension check, press Enter.")
    return {}


## Quiz and answer capture
Create a quiz and collect the answer.

In [ ]:
def generate_quiz(state: Dict[str, Any]) -> Dict[str, Any]:
    summary = state.get("summary") or ""

    prompt = (
        "You are creating a single-question quiz based ONLY on the summary below. "
        "Provide one clear, answerable question. Do not include the answer.\n\n"
        f"SUMMARY:\n{summary}"
    )

    response = llm.invoke(prompt)
    question = (getattr(response, "content", None) or "").strip()

    return {
        "quiz_question": question,
        "messages": append_message(state, "assistant", f"Quiz question: {question}"),
    }


def ask_quiz_answer(state: Dict[str, Any]) -> Dict[str, Any]:
    question = state.get("quiz_question") or ""
    print("\nQuiz Question:\n")
    print(question)
    answer = get_user_input("\nYour answer: ").strip()
    return {
        "quiz_answer": answer,
        "messages": append_message(state, "user", f"Quiz answer: {answer}"),
    }


## Grading and feedback
Assess the answer with citations.

In [ ]:
def _get_citations_text(state: Dict[str, Any]) -> str:
    """Self-contained citation formatter (no external dependency)."""
    results = state.get("search_results", []) or []
    if not results:
        return "No citations available."
    lines = []
    for i, item in enumerate(results, start=1):
        url = item.get("url") or ""
        title = item.get("title") or item.get("source") or ""
        if not title:
            content = (item.get("content") or "").strip()
            if content:
                words = content.split()
                title = " ".join(words[:10]) + ("..." if len(words) > 10 else "")
            else:
                title = url or f"Source {i}"
        lines.append(f"[{i}] {title}\n{url}")
    return "\n\n".join(lines)


def grade_answer(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    Lenient grading; returns ONLY:
      - grade: str (single letter A-F, or 'N/A' if parsing failed)
      - grade_explanation: str (a block containing 'Grade:' and 'Feedback:' + appended citations)
    """
    summary = state.get("summary") or ""
    question = state.get("quiz_question") or ""
    answer = state.get("quiz_answer") or ""

    prompt = (
        "You are grading a patient's answer using ONLY the summary provided. "
        "Be LENIENT and supportive:\n"
        "- If the answer captures the main idea or is mostly correct, award at least a 'B'.\n"
        "- Use 'C' or below only if the answer is clearly incorrect, contradicts the summary, "
        "  or misses the key point.\n"
        "- When in doubt, choose the higher grade.\n\n"
        "Respond in plain English (no JSON). "
        "Format strictly as:\n"
        "Grade: <A|B|C|D|F>\n"
        "Feedback: <2-5 concise sentences citing the summary with [1], [2], etc.>\n\n"
        f"SUMMARY:\n{summary}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}"
    )

    response = llm.invoke(prompt)
    feedback_block = (getattr(response, "content", None) or "").strip()

    grade_match = re.search(r"Grade\s*[:\-]\s*([A-F])", feedback_block, re.IGNORECASE)
    grade = grade_match.group(1).upper() if grade_match else "N/A"

    citations_text = _get_citations_text(state)

    if "Sources:" in feedback_block:
        feedback_with_citations = feedback_block
    else:
        if "Feedback:" in feedback_block:
            feedback_with_citations = feedback_block.rstrip() + "\n\nSources:\n" + citations_text
        else:
            feedback_lines = [
                f"Grade: {grade}",
                "Feedback: See sources below.\n",
                "Sources:",
                citations_text,
            ]
            feedback_with_citations = "\n".join(feedback_lines)

    return {
        "grade": grade,
        "grade_explanation": feedback_with_citations,
    }


def present_grade(state: HealthBotState) -> Dict[str, Any]:
    print("\nQuiz Result:\n")
    print(state.get("grade_explanation") or "")
    return {}


## Restart and reset
Offer another topic and clear state.

In [ ]:
def ask_restart(state: Dict[str, Any]) -> Dict[str, Any]:
    while True:
        choice = get_user_input("Would you like to learn about another topic? (yes/no)").strip().lower()

        if choice in {"yes", "y"}:
            return {"restart": True}

        if choice in {"no", "n"}:
            print("Thanks for using the bot.")
            return {"restart": False}

        print_panel("Please answer 'yes' or 'no'.", title="Invalid Input", style="red")


def reset_state(_: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "topic": None,
        "search_results": None,
        "summary": None,
        "quiz_question": None,
        "quiz_answer": None,
        "grade": None,
        "grade_explanation": None,
        "messages": [],
        "restart": None,
    }


## Routing helper
Decide whether to continue or end.

In [ ]:
# Routing handled in graph wiring.


## Graph wiring
Connect nodes and compile the app.

In [ ]:
graph = StateGraph(HealthBotState)

graph.add_node("ask_topic", ask_topic)
graph.add_node("search_topic", search_topic)
graph.add_node("summarize_results", summarize_results)
graph.add_node("present_summary", present_summary)
graph.add_node("wait_for_ready", wait_for_ready)
graph.add_node("generate_quiz", generate_quiz)
graph.add_node("ask_quiz_answer", ask_quiz_answer)
graph.add_node("grade_answer", grade_answer)
graph.add_node("present_grade", present_grade)
graph.add_node("ask_restart", ask_restart)
graph.add_node("reset_state", reset_state)

graph.set_entry_point("ask_topic")
graph.add_edge("ask_topic", "search_topic")
graph.add_edge("search_topic", "summarize_results")
graph.add_edge("summarize_results", "present_summary")
graph.add_edge("present_summary", "wait_for_ready")
graph.add_edge("wait_for_ready", "generate_quiz")
graph.add_edge("generate_quiz", "ask_quiz_answer")
graph.add_edge("ask_quiz_answer", "grade_answer")
graph.add_edge("grade_answer", "present_grade")
graph.add_edge("present_grade", "ask_restart")

graph.add_conditional_edges(
    "ask_restart",
    lambda state: "restart" if state.get("restart") else "end",
    {
        "restart": "reset_state",
        "end": END,
    },
)

graph.add_edge("reset_state", "ask_topic")

app = graph.compile()


## Run the app
Invoke the workflow.

In [ ]:
print("HealthBot is ready. Follow the prompts below.\n")
app.invoke({"messages": []})